# Step 3 Confounds Study With A Nonlinear Probe

This notebook stays close to the original Step 3 study.

It does four things:
- read the saved Step 3 panel and latent matrix,
- analyze the Step 3 ridge results and confound tables,
- fit an MLP probe on the same raw and residual targets,
- compare the MLP results back to the original Step 3 ridge results.


## 1) Setup

The defaults below are based on the Step 3 artifacts: we exclude the very confounded size-driven properties and keep the lower- and mid-`R^2` properties that are more interesting for a nonlinear probe.


In [ ]:
from pathlib import Path
import copy
import json
import random
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import r2_score
from tqdm.auto import tqdm

SEED = 42
BATCH_SIZE = 8192
MAX_EPOCHS = 15
MIN_EPOCHS = 6
PATIENCE = 4
MIN_VAL_R2_IMPROVEMENT = 1e-3
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_WIDTH = 256
EXPECTED_SPLIT_SIZES = {'train': 635522, 'val': 79440, 'test': 79441}

EXCLUDED_CONFOUNDED_PROPERTIES = [
    'HeavyAtomCount',
    'MolWt',
    'ExactMolWt',
    'BertzCT',
    'RingCount',
]

STEP3_PRIORITY_PROPERTIES = [
    'cLogP',
    'TPSA',
    'FractionCSP3',
    'HBA',
    'HBD',
    'AromaticRingCount',
    'QED',
    'NumSpiroAtoms',
    'NumBridgeheadAtoms',
]

MANUAL_TARGET_PROPERTIES = None
# Example:
# MANUAL_TARGET_PROPERTIES = ['QED', 'HBD', 'NumSpiroAtoms', 'NumBridgeheadAtoms']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = bool(torch.cuda.is_available())
print('device =', device)
print('seed =', SEED)
print('batch_size =', BATCH_SIZE)
print('max_epochs =', MAX_EPOCHS)
print('use_amp =', USE_AMP)


## 2) Load Existing Step 3 Artifacts

Only Step 3 artifacts are used here for study design and ridge comparison.


In [ ]:
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != '_patched_notebooks':
    NOTEBOOK_DIR = Path('artifacts/model_compare/nat_model_h256_l256/_patched_notebooks').resolve()

MODEL_DIR = NOTEBOOK_DIR.parent
STEP3_DIR = MODEL_DIR / 'confounds_step3'
OUTPUT_DIR = MODEL_DIR / 'confounds_mlp_step3'
TABLES_DIR = OUTPUT_DIR / 'tables'
PLOTS_DIR = OUTPUT_DIR / 'plots'
MODEL_EXPORT_DIR = OUTPUT_DIR / 'mlp_models'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

RAW_MODEL_REGISTRY = {}
RESID_MODEL_REGISTRY = {}

panel = pd.read_csv(STEP3_DIR / 'panels_step3_with_residuals.csv')
z_mu = np.load(STEP3_DIR / 'Z_mu.npy').astype(np.float32)
step3_raw = pd.read_csv(STEP3_DIR / 'tables' / 'r2_Z_to_Y.csv')
step3_resid = pd.read_csv(STEP3_DIR / 'tables' / 'r2_Z_to_Y_vs_Yresid.csv')
r2_z_to_c = pd.read_csv(STEP3_DIR / 'tables' / 'r2_Z_to_C.csv')
top20_confounded = pd.read_csv(STEP3_DIR / 'tables' / 'top20_confounded_pairs_YC.csv')
corr_spearman = pd.read_csv(STEP3_DIR / 'tables' / 'corr_spearman_YC.csv', index_col=0)
corr_pearson = pd.read_csv(STEP3_DIR / 'tables' / 'corr_pearson_YC.csv', index_col=0)

assert len(panel) == len(z_mu), (len(panel), len(z_mu))
print('panel rows =', len(panel))
print('latent shape =', z_mu.shape)
print('step3 dir =', STEP3_DIR)


## 3) Analyze The Original Step 3 Ridge Study

Before fitting anything nonlinear, look at the original linear probe results and the confound correlations.


In [ ]:
max_abs_spearman = corr_spearman.abs().max(axis=1).rename('max_abs_spearman').reset_index().rename(columns={'index': 'property'})
max_abs_pearson = corr_pearson.abs().max(axis=1).rename('max_abs_pearson').reset_index().rename(columns={'index': 'property'})

step3_summary = (
    step3_raw[['property', 'r2_val', 'r2_test']]
    .merge(step3_resid[['property', 'r2_residual', 'r2_residual_test', 'delta_test']], on='property', how='left')
    .merge(max_abs_spearman, on='property', how='left')
    .merge(max_abs_pearson, on='property', how='left')
)

step3_summary['exclude_as_very_confounded'] = step3_summary['property'].isin(EXCLUDED_CONFOUNDED_PROPERTIES)
step3_summary['in_priority_set'] = step3_summary['property'].isin(STEP3_PRIORITY_PROPERTIES)
step3_summary = step3_summary.sort_values('r2_test', ascending=False).reset_index(drop=True)
step3_summary.to_csv(TABLES_DIR / 'step3_linear_probe_summary_for_mlp.csv', index=False)

if MANUAL_TARGET_PROPERTIES is None:
    TARGET_PROPERTIES = [
        prop for prop in STEP3_PRIORITY_PROPERTIES
        if prop in step3_summary['property'].tolist() and prop not in EXCLUDED_CONFOUNDED_PROPERTIES
    ]
else:
    TARGET_PROPERTIES = list(MANUAL_TARGET_PROPERTIES)

print('Step 3 confounds are highly predictable from Z:')
display(r2_z_to_c.sort_values('r2_test', ascending=False))

print('Top Step 3 confounded property/confound pairs:')
display(top20_confounded.head(20))

print('Step 3 property summary:')
display(step3_summary[['property', 'r2_test', 'r2_residual_test', 'delta_test', 'max_abs_spearman', 'exclude_as_very_confounded', 'in_priority_set']])

print('Default MLP study properties:')
print(TARGET_PROPERTIES)


## 4) Recreate The Step 3 Split And Scaling Discipline

Step 3 used train-only standardization inside each fit. Since no reusable Step 3 scaler artifact was saved, we rebuild that logic directly from the saved Step 3 panel and latent matrix.


In [ ]:
split = panel['split'].astype(str).str.lower().to_numpy()
split_sizes = {name: int((split == name).sum()) for name in ('train', 'val', 'test')}
assert split_sizes == EXPECTED_SPLIT_SIZES, split_sizes

if 'is_rdkit_valid' in panel.columns:
    valid_mask = panel['is_rdkit_valid'].fillna(False).to_numpy(dtype=bool)
else:
    valid_mask = np.ones(len(panel), dtype=bool)

train_mask = (split == 'train') & valid_mask
val_mask = (split == 'val') & valid_mask
test_mask = (split == 'test') & valid_mask

z_mean = z_mu[train_mask].mean(axis=0).astype(np.float32)
z_std = z_mu[train_mask].std(axis=0).astype(np.float32)
z_std = np.where(z_std < 1e-8, 1.0, z_std).astype(np.float32)

X_train = ((z_mu[train_mask] - z_mean) / z_std).astype(np.float32)
X_val = ((z_mu[val_mask] - z_mean) / z_std).astype(np.float32)
X_test = ((z_mu[test_mask] - z_mean) / z_std).astype(np.float32)

panel_train = panel.loc[train_mask].reset_index(drop=True)
panel_val = panel.loc[val_mask].reset_index(drop=True)
panel_test = panel.loc[test_mask].reset_index(drop=True)

print('split sizes =', split_sizes)
print('valid train/val/test =', int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum()))
print('latent dim =', X_train.shape[1])


## 5) MLP Helper Functions

The batch-level `tqdm` is the main progress signal here, so you can watch the optimizer move through each epoch.


In [ ]:
def safe_r2(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 2:
        return np.nan
    if np.std(y_true[mask]) < 1e-12:
        return np.nan
    return float(r2_score(y_true[mask], y_pred[mask]))

def train_target_stats(y_train_full):
    finite = np.isfinite(y_train_full)
    if finite.sum() == 0:
        return 0.0, 1.0
    mean = float(np.nanmean(y_train_full[finite]))
    std = float(np.nanstd(y_train_full[finite]))
    if not np.isfinite(std) or std < 1e-8:
        std = 1.0
    return mean, std

class MLPProbe(torch.nn.Module):
    def __init__(self, input_dim, hidden_width=HIDDEN_WIDTH):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_width),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_width, hidden_width),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_width, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def predict_raw(model, x_np, y_mean, y_std, batch_size=BATCH_SIZE):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(x_np), batch_size):
            stop = min(start + batch_size, len(x_np))
            xb = torch.from_numpy(x_np[start:stop]).to(device, non_blocking=True)
            pred_std = model(xb)
            pred_raw = pred_std * y_std + y_mean
            outputs.append(pred_raw.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)

def fit_mlp_target(target_name, y_train_full, y_val_full, y_test_full, registry=None):
    train_finite = np.isfinite(y_train_full)
    val_finite = np.isfinite(y_val_full)
    test_finite = np.isfinite(y_test_full)

    xtr = X_train[train_finite]
    xva = X_val[val_finite]
    xte = X_test[test_finite]

    ytr_raw = y_train_full[train_finite].astype(np.float32)
    yva_raw = y_val_full[val_finite].astype(np.float32)
    yte_raw = y_test_full[test_finite].astype(np.float32)

    y_mean, y_std = train_target_stats(y_train_full)
    ytr_std = ((ytr_raw - y_mean) / y_std).astype(np.float32)

    dataset = torch.utils.data.TensorDataset(torch.from_numpy(xtr), torch.from_numpy(ytr_std))
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        pin_memory=torch.cuda.is_available(),
    )

    model = MLPProbe(X_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = torch.nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_state = None
    best_val_r2 = -np.inf
    best_epoch = -1
    bad_epochs = 0
    t0 = perf_counter()

    for epoch in range(MAX_EPOCHS):
        model.train()
        running_loss = 0.0
        sample_count = 0
        batch_iter = tqdm(loader, total=len(loader), leave=False, mininterval=0.5, desc=f'{target_name} | epoch {epoch + 1}/{MAX_EPOCHS}')
        for xb, yb in batch_iter:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred = model(xb)
                loss = loss_fn(pred, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            batch_n = int(yb.shape[0])
            running_loss += float(loss.detach().cpu()) * batch_n
            sample_count += batch_n
            batch_iter.set_postfix(loss=f'{float(loss.detach().cpu()):.4f}')

        val_pred = predict_raw(model, xva, y_mean, y_std)
        val_r2 = safe_r2(yva_raw, val_pred)
        mean_train_loss = running_loss / max(sample_count, 1)
        print(f'epoch {epoch + 1:02d} | {target_name} | train_loss={mean_train_loss:.4f} | val_r2={val_r2:.4f}')

        improved = np.isfinite(val_r2) and (val_r2 > best_val_r2 + MIN_VAL_R2_IMPROVEMENT)
        if improved:
            best_val_r2 = val_r2
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch + 1 >= MIN_EPOCHS and bad_epochs >= PATIENCE:
            print(f'early stop on {target_name} at epoch {epoch + 1}')
            break

    if best_state is None:
        raise RuntimeError(f'No valid model state for {target_name}')

    model.load_state_dict(best_state)
    train_pred = predict_raw(model, xtr, y_mean, y_std)
    val_pred = predict_raw(model, xva, y_mean, y_std)
    test_pred = predict_raw(model, xte, y_mean, y_std)
    elapsed_seconds = perf_counter() - t0

    result = {
        'target': target_name,
        'best_epoch': int(best_epoch + 1),
        'elapsed_seconds': float(elapsed_seconds),
        'y_mean_train': float(y_mean),
        'y_std_train': float(y_std),
        'n_train': int(train_finite.sum()),
        'n_val': int(val_finite.sum()),
        'n_test': int(test_finite.sum()),
        'r2_train': safe_r2(ytr_raw, train_pred),
        'r2_val': safe_r2(yva_raw, val_pred),
        'r2_test': safe_r2(yte_raw, test_pred),
    }

    if registry is not None:
        registry[str(target_name)] = {
            'state_dict': {k: v.detach().cpu().clone() for k, v in best_state.items()},
            'input_dim': int(X_train.shape[1]),
            'hidden_width': int(HIDDEN_WIDTH),
            'target_name': str(target_name),
            'y_mean_train': float(y_mean),
            'y_std_train': float(y_std),
            'best_epoch': int(result['best_epoch']),
            'r2_val': float(result['r2_val']),
            'r2_test': float(result['r2_test']),
        }
    print(f"finished {target_name}: best_epoch={result['best_epoch']}, val_r2={result['r2_val']:.4f}, test_r2={result['r2_test']:.4f}, seconds={result['elapsed_seconds']:.1f}")
    return result


## 6) Fit MLP On Raw Properties

This directly replaces the original Step 3 ridge `Z -> Y` probe on the selected target set.


In [ ]:
raw_rows = []
for prop in TARGET_PROPERTIES:
    print('\n' + '=' * 80)
    print(f'starting raw target: {prop}')
    raw_rows.append(
        fit_mlp_target(
            prop,
            pd.to_numeric(panel_train[prop], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_val[prop], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_test[prop], errors='coerce').to_numpy(dtype=float),
            registry=RAW_MODEL_REGISTRY,
        )
    )

mlp_raw_df = pd.DataFrame(raw_rows).sort_values('r2_test', ascending=False).reset_index(drop=True)
mlp_raw_df.to_csv(TABLES_DIR / 'r2_Z_to_Y_mlp.csv', index=False)
display(mlp_raw_df)


## 7) Fit MLP On Residual Properties

This directly replaces the original Step 3 ridge `Z -> Y_resid` probe on the same target set.


In [ ]:
resid_rows = []
for prop in TARGET_PROPERTIES:
    target_name = f'resid_{prop}'
    print('\n' + '=' * 80)
    print(f'starting residual target: {target_name}')
    resid_rows.append(
        fit_mlp_target(
            target_name,
            pd.to_numeric(panel_train[target_name], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_val[target_name], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_test[target_name], errors='coerce').to_numpy(dtype=float),
            registry=RESID_MODEL_REGISTRY,
        )
    )

mlp_resid_df = pd.DataFrame(resid_rows)
mlp_resid_df['property'] = mlp_resid_df['target'].str.replace('^resid_', '', regex=True)
mlp_resid_df = mlp_resid_df.sort_values('r2_test', ascending=False).reset_index(drop=True)
mlp_resid_df.to_csv(TABLES_DIR / 'r2_Z_to_Yresid_mlp.csv', index=False)
display(mlp_resid_df)


## 8) Compare MLP To The Original Step 3 Ridge Results

The comparison here uses the Step 3 saved ridge tables only.


In [ ]:
raw_compare = mlp_raw_df.merge(
    step3_raw[['property', 'r2_val', 'r2_test']].rename(columns={'r2_val': 'ridge_r2_val', 'r2_test': 'ridge_r2_test'}),
    left_on='target',
    right_on='property',
    how='left',
)
raw_compare['delta_val_mlp_minus_ridge'] = raw_compare['r2_val'] - raw_compare['ridge_r2_val']
raw_compare['delta_test_mlp_minus_ridge'] = raw_compare['r2_test'] - raw_compare['ridge_r2_test']
raw_compare.to_csv(TABLES_DIR / 'mlp_vs_ridge_raw.csv', index=False)

resid_compare = mlp_resid_df.merge(
    step3_resid[['property', 'r2_residual', 'r2_residual_test', 'delta_test']].rename(columns={'r2_residual': 'ridge_r2_val', 'r2_residual_test': 'ridge_r2_test'}),
    on='property',
    how='left',
)
resid_compare['delta_val_mlp_minus_ridge'] = resid_compare['r2_val'] - resid_compare['ridge_r2_val']
resid_compare['delta_test_mlp_minus_ridge'] = resid_compare['r2_test'] - resid_compare['ridge_r2_test']
resid_compare.to_csv(TABLES_DIR / 'mlp_vs_ridge_resid.csv', index=False)

raw_vs_resid = (
    mlp_raw_df[['target', 'r2_val', 'r2_test']]
    .rename(columns={'target': 'property', 'r2_val': 'mlp_raw_r2_val', 'r2_test': 'mlp_raw_r2_test'})
    .merge(
        mlp_resid_df[['property', 'r2_val', 'r2_test']].rename(columns={'r2_val': 'mlp_resid_r2_val', 'r2_test': 'mlp_resid_r2_test'}),
        on='property',
        how='left',
    )
)
raw_vs_resid['delta_test_resid_minus_raw'] = raw_vs_resid['mlp_resid_r2_test'] - raw_vs_resid['mlp_raw_r2_test']
raw_vs_resid.to_csv(TABLES_DIR / 'mlp_raw_vs_resid.csv', index=False)

display(raw_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False))
display(resid_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False))
display(raw_vs_resid.sort_values('delta_test_resid_minus_raw', ascending=False))


## 9) Quick Visual Summary

A compact view of where the nonlinear probe helps and how raw vs residual behavior changes.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

top_raw = raw_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False)
axes[0].barh(top_raw['target'], top_raw['delta_test_mlp_minus_ridge'])
axes[0].invert_yaxis()
axes[0].set_title('Raw: MLP - Ridge (test R^2)')

top_resid = resid_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False)
axes[1].barh(top_resid['target'], top_resid['delta_test_mlp_minus_ridge'])
axes[1].invert_yaxis()
axes[1].set_title('Residual: MLP - Ridge (test R^2)')

gap = raw_vs_resid.sort_values('delta_test_resid_minus_raw', ascending=False)
axes[2].barh(gap['property'], gap['delta_test_resid_minus_raw'])
axes[2].invert_yaxis()
axes[2].set_title('MLP: Residual - Raw (test R^2)')

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'mlp_probe_summary.png', dpi=220)
plt.show()


## 10) Delta R^2 Comparison

Compute the raw and residual `\Delta R^2 = R^2_{MLP} - R^2_{Ridge}` values side by side for direct comparison.


In [ ]:
delta_r2_df = (
    raw_compare[['property', 'target', 'r2_test', 'ridge_r2_test', 'delta_test_mlp_minus_ridge']]
    .rename(columns={
        'target': 'raw_target',
        'r2_test': 'raw_mlp_r2_test',
        'ridge_r2_test': 'raw_ridge_r2_test',
        'delta_test_mlp_minus_ridge': 'raw_delta_r2_mlp_minus_ridge',
    })
    .merge(
        resid_compare[['property', 'target', 'r2_test', 'ridge_r2_test', 'delta_test_mlp_minus_ridge']]
        .rename(columns={
            'target': 'resid_target',
            'r2_test': 'resid_mlp_r2_test',
            'ridge_r2_test': 'resid_ridge_r2_test',
            'delta_test_mlp_minus_ridge': 'resid_delta_r2_mlp_minus_ridge',
        }),
        on='property',
        how='left',
    )
    .merge(step3_summary[['property', 'max_abs_spearman']], on='property', how='left')
)

delta_r2_df['resid_minus_raw_delta_r2'] = (
    delta_r2_df['resid_delta_r2_mlp_minus_ridge'] - delta_r2_df['raw_delta_r2_mlp_minus_ridge']
)

delta_r2_df = delta_r2_df.sort_values('property').reset_index(drop=True)
delta_r2_df.to_csv(TABLES_DIR / 'mlp_ridge_delta_r2_comparison.csv', index=False)

display(delta_r2_df[['property', 'raw_delta_r2_mlp_minus_ridge', 'resid_delta_r2_mlp_minus_ridge', 'resid_minus_raw_delta_r2', 'max_abs_spearman']])


## 11) Save Final Summary

This saves a compact study summary next to the tables and plots.


In [ ]:
summary = {
    'model_dir': str(MODEL_DIR),
    'step3_dir': str(STEP3_DIR),
    'output_dir': str(OUTPUT_DIR),
    'device': str(device),
    'batch_size': int(BATCH_SIZE),
    'max_epochs': int(MAX_EPOCHS),
    'min_epochs': int(MIN_EPOCHS),
    'patience': int(PATIENCE),
    'min_val_r2_improvement': float(MIN_VAL_R2_IMPROVEMENT),
    'use_amp': bool(USE_AMP),
    'target_properties': list(TARGET_PROPERTIES),
    'excluded_confounded_properties': list(EXCLUDED_CONFOUNDED_PROPERTIES),
    'top_raw_test_gains': raw_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False).head(5)[['target', 'delta_test_mlp_minus_ridge', 'r2_test', 'ridge_r2_test']].to_dict(orient='records'),
    'top_resid_test_gains': resid_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False).head(5)[['target', 'delta_test_mlp_minus_ridge', 'r2_test', 'ridge_r2_test']].to_dict(orient='records'),
}

with open(OUTPUT_DIR / 'summary_mlp_confounds.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

summary


## 11) Save In-Memory MLP Weights If Available

Run this after the raw and residual MLP cells. If checkpoints were kept in memory during training, they are exported for later traversal work.


In [ ]:
saved_rows = []

for variant_name, registry in [('raw', globals().get('RAW_MODEL_REGISTRY', {})), ('residual', globals().get('RESID_MODEL_REGISTRY', {}))]:
    if not registry:
        print(f'No in-memory {variant_name} MLP checkpoints found.')
        continue

    for target_name, bundle in registry.items():
        out_path = MODEL_EXPORT_DIR / f'{target_name}.pt'
        torch.save(bundle, out_path)
        saved_rows.append(
            {
                'variant': variant_name,
                'target_name': target_name,
                'path': str(out_path),
                'best_epoch': int(bundle['best_epoch']),
                'r2_val': float(bundle['r2_val']),
                'r2_test': float(bundle['r2_test']),
            }
        )

if saved_rows:
    checkpoint_manifest = pd.DataFrame(saved_rows).sort_values(['variant', 'target_name']).reset_index(drop=True)
    checkpoint_manifest.to_csv(TABLES_DIR / 'mlp_checkpoint_manifest.csv', index=False)
    display(checkpoint_manifest)
else:
    print('No checkpoints were exported. If this notebook was run before the registry patch, rerun the raw/residual MLP cells and then rerun this cell.')
